#### Documentation:
Only 4 quarters of data, tracking Q1 2025 - Q1 2026. Narrow window that can only really find a correlation, but not establish a real relationship due to free tier limitation.
"Are heavy lobbyists currently beating earnings estimates more consistently than light lobbyists?"

In [2]:
import finnhub
import os
import time
from collections import defaultdict
from dotenv import load_dotenv
load_dotenv()

finnhub_client = finnhub.Client(api_key=os.getenv("FINNHUB_API_KEY"))

tickers = [
    # DEFENSE - High Lobby
    "LMT", "RTX", "NOC", "GD", "BA", "LHX", "LDOS", "HII", "BAESY", "SAIC",
    # DEFENSE - Low Lobby
    "TXT", "TDG", "HEI", "DRS", "KTOS", "AVAV", "MRCY", "CW", "MOG.A", "DCO",
    # ENERGY - High Lobby
    "XOM", "CVX", "COP", "OXY", "BP", "NEE", "D", "DUK", "HAL", "BKR",
    # ENERGY - Low Lobby
    "SLB", "VLO", "PSX", "EOG", "FANG", "DVN", "CTRA", "AR", "CHRD", "MTDR",
    # TECH - High Lobby
    "MSFT", "AMZN", "GOOGL", "IBM", "ORCL", "PLTR", "BAH", "CACI", "PSN", "CRM",
    # TECH - Low Lobby
    "AAPL", "META", "NVDA", "CSCO", "PANW", "CRWD", "SNOW", "DDOG", "NET", "TWLO"
]

results = {}
errors = []

# Field-level tracking across all tickers
field_present   = defaultdict(int)  # how many tickers returned this field at all
field_null      = defaultdict(int)  # how many tickers had this field as None or ""
field_types     = defaultdict(set)  # what Python types each field returned

for i, ticker in enumerate(tickers):
    try:
        data = finnhub_client.company_profile2(symbol=ticker)

        if not data:
            errors.append((ticker, "empty response"))
            results[ticker] = {}
        else:
            results[ticker] = data
            for field, value in data.items():
                field_present[field] += 1
                if value is None or value == "":
                    field_null[field] += 1
                else:
                    field_types[field].add(type(value).__name__)

    except Exception as e:
        errors.append((ticker, f"API error: {str(e)}"))
        results[ticker] = {}

    if i < len(tickers) - 1:
        time.sleep(1)

# ─────────────────────────────────────────
# SUMMARY
# ─────────────────────────────────────────
total         = len(tickers)
successful    = len([r for r in results.values() if r])
empty_tickers = [t for t, r in results.items() if not r]

print(f"✅ Successfully pulled: {successful} / {total}")
print(f"❌ Errors:              {len(errors)}")
print(f"📭 Empty responses:     {empty_tickers if empty_tickers else 'None'}\n")

if errors:
    print("─── Errors ───")
    for ticker, msg in errors:
        print(f"   {ticker}: {msg}")
    print()

# ─────────────────────────────────────────
# PYDANTIC MODEL DECISION TABLE
# ─────────────────────────────────────────
print("─── Pydantic Model Decision Table ───")
print(f"{'Field':<25} {'Present':>10} {'Null/Empty':>12} {'Types':<20} {'Recommendation'}")
print("─" * 90)

for field in sorted(field_present.keys()):
    present    = field_present[field]
    null_count = field_null.get(field, 0)
    types      = ", ".join(field_types.get(field, {"unknown"}))
    always_present = present == successful
    ever_null      = null_count > 0

    if not always_present:
        recommendation = "Optional  ← missing from some tickers"
    elif ever_null:
        recommendation = "Optional  ← null in some tickers"
    else:
        recommendation = "Required  ← always present and never null"

    print(f"{field:<25} {present:>7}/{total}  {null_count:>8} null   {types:<20} {recommendation}")

✅ Successfully pulled: 60 / 60
❌ Errors:              0
📭 Empty responses:     None

─── Pydantic Model Decision Table ───
Field                        Present   Null/Empty Types                Recommendation
──────────────────────────────────────────────────────────────────────────────────────────
country                        60/60         0 null   str                  Required  ← always present and never null
currency                       60/60         0 null   str                  Required  ← always present and never null
estimateCurrency               60/60         0 null   str                  Required  ← always present and never null
exchange                       60/60         0 null   str                  Required  ← always present and never null
finnhubIndustry                60/60         0 null   str                  Required  ← always present and never null
ipo                            60/60         0 null   str                  Required  ← always present and never nul